# 4. Forward kinematics

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

Forward kinematics answers one question: given every joint's local axis-angle, where are the 16 joint positions.

The rest positions are computed elsewhere and written `J[0]` through `J[15]`. Chapter 6 shows that `J` comes from the vertices. This chapter treats `J` as 16 known points.

## The formula

The root:

```text
R_0 = Rodrigues(pose_0)
p_0 = J[0]
```

Every other joint, with parent `parent`:

```text
R_i     = R_parent @ Rodrigues(pose_i)
offset  = J[i] - J[parent]
p_i     = p_parent + R_parent @ offset
```

`R_i` is the orientation of joint `i` in the world. `p_i` is its position.

The position line uses `R_parent`. The joint's own rotation is applied after its position is known. It changes the bone's orientation, so children and the skin attached to it move. It does not push its own joint centre.

So:

- Rotating only the index PIP leaves the index MCP and the wrist where they are, and moves the index DIP.
- Rotating only the wrist moves all five fingers, because every parent chain passes through the wrist.
- A finger rotation does not carry the wrist somewhere else.

The next cell checks this on the teaching skeleton. Those bone lengths are millimetres drawn for the picture. They are not the official template.


In [18]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import mano_skeleton as sk

rest, _ = sk.forward_kinematics(sk.zero_pose())
bent = sk.zero_pose()
bent[1] = np.array([0.0, 0.0, np.deg2rad(30.0)])
moved, _ = sk.forward_kinematics(bent)

print(f"index PIP moved {np.linalg.norm(moved[2] - rest[2]):.3e} mm")
print(f"index DIP moved {np.linalg.norm(moved[3] - rest[3]):.3f} mm")
print(f"wrist moved     {np.linalg.norm(moved[0] - rest[0]):.3e} mm")
assert np.allclose(moved[1], rest[1])
assert np.linalg.norm(moved[3] - rest[3]) > 1.0
assert np.allclose(moved[0], rest[0])


index PIP moved 2.073e+01 mm
index DIP moved 33.165 mm
wrist moved     0.000e+00 mm


In [7]:
sk.forward_kinematics(sk.zero_pose())

(array([[  0.,   0.,   0.],
        [-20.,  88.,   4.],
        [-22., 128.,   4.],
        [-23., 152.,   4.],
        [  0.,  92.,   2.],
        [  0., 136.,   2.],
        [  0., 162.,   2.],
        [ 36.,  72.,   0.],
        [ 38., 104.,   0.],
        [ 39., 122.,   0.],
        [ 18.,  84.,   2.],
        [ 19., 124.,   2.],
        [ 20., 148.,   2.],
        [-28.,  30.,  14.],
        [-54.,  46.,  24.],
        [-74.,  52.,  30.]]),
 array([[[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],
 
        [[1., 0., 0.],
         [0., 1., 0.],


## The same fact on the official hand

The official template's axes wait until chapter 10. Measure one result now: on the flat template, rotate only the index MCP by 30 degrees about the model's Z axis.

The index MCP should stay put. The index DIP and the index tip should move toward -Y. The wrist should stay put.


In [ ]:
from official_mano import require_model

model = require_model()
flat = model.forward(flat_hand=True)
hand_pose = np.zeros(45)
hand_pose[2] = np.deg2rad(30.0)  # rz of the index MCP
curled = model.forward(hand_pose=hand_pose, flat_hand=True)

def moved_mm(index):
    return (curled["joints"][index] - flat["joints"][index]) * 1000
print("index MCP displacement mm", np.round(moved_mm(1), 3))
print("index DIP displacement mm", np.round(np.linalg.norm(moved_mm(3)), 2))
print("index tip displacement mm ", np.round((curled["tips"]["index"] - flat["tips"]["index"]) * 1000, 2))
print("wrist displacement mm     ", np.round(moved_mm(0), 4))
assert np.allclose(moved_mm(1), 0.0, atol=1e-6)
assert np.allclose(moved_mm(0), 0.0, atol=1e-6)
assert moved_mm(3)[1] < -20


index MCP displacement mm [0. 0. 0.]
index DIP displacement mm 28.42
index tip displacement mm  [  8.82 -40.68  -0.  ]
wrist displacement mm      [0. 0. 0.]


The Y component of the displacement is negative, so the tip travels toward -Y. That is the same fact as "flexion about +Z points toward the palm". Chapter 10 states the axes once, with the measurements beside them.

## Global rotation is separate from the local ones

`pose_0` is `global_orient`. It sits on the left and multiplies into every child's orientation. Turning the whole hand by one angle leaves every distance between joints unchanged. That is a rigid motion: rotation and translation, with no stretch.

`transl` is not in the three lines above. After forward kinematics, the same vector is added to the 16 joints and the 778 vertices.


In [3]:
turned = model.forward(global_orient=np.array([0.3, -0.2, 0.4]), flat_hand=True)
pairs = [(1, 4), (4, 7), (13, 15), (0, 6)]
for a, b in pairs:
    before = np.linalg.norm(flat["joints"][a] - flat["joints"][b])
    after = np.linalg.norm(turned["joints"][a] - turned["joints"][b])
    print(f"joint {a} to {b}: {before * 1000:.2f} mm → {after * 1000:.2f} mm")
    assert np.isclose(before, after)


joint 1 to 4: 25.20 mm → 25.20 mm
joint 4 to 7: 48.26 mm → 48.26 mm
joint 13 to 15: 56.86 mm → 56.86 mm
joint 0 to 6: 149.31 mm → 149.31 mm


## Check

In one sentence: the index DIP rotates by 40 degrees. Why does the index PIP stay where it is?

The DIP rotation appears only in `R_DIP`. The PIP position is fixed by the rotation of the PIP's parent (the index MCP) and by the PIP's rest offset. The DIP is a child of the PIP. A child's rotation is not written into the parent's position.

The next chapter puts forward kinematics back into the full forward formula and looks at the steps in front of it.
